# Chapter 20: Confidence Intervals
This notebook uses synthetic Nusantara Rasa Global data. Set seeds make every result reproducible. Run the completed cells in order.

In [1]:
import math
import random
import statistics
import matplotlib.pyplot as plt
from datasciencebook.confidence_intervals import normal_interval, wilson_interval, bootstrap_mean_interval
rng = random.Random(20260829)
population = [rng.lognormvariate(math.log(120), 0.25) for _ in range(400)]
true_mean = statistics.mean(population)
print(f"Population stores: {len(population)}")
print(f"True population mean: {true_mean:.2f}")

Population stores: 400
True population mean: 124.69


In [2]:
sample = rng.sample(population, 40)
sample_mean = statistics.mean(sample)
estimated_se = statistics.stdev(sample) / math.sqrt(len(sample))
mean_interval = normal_interval(sample_mean, estimated_se)
print(f"Sample mean: {sample_mean:.2f}")
print(f"Estimated standard error: {estimated_se:.2f}")
print(f"95% normal interval: [{mean_interval[0]:.2f}, {mean_interval[1]:.2f}]")

Sample mean: 133.22
Estimated standard error: 4.31
95% normal interval: [124.76, 141.68]


In [3]:
late, shipments = 18, 200
late_interval = wilson_interval(late, shipments)
print(f"Late-shipment estimate: {100 * late / shipments:.1f}%")
print(f"95% Wilson interval: [{100 * late_interval[0]:.1f}%, {100 * late_interval[1]:.1f}%]")

Late-shipment estimate: 9.0%
95% Wilson interval: [5.8%, 13.8%]


In [4]:
bootstrap_interval = bootstrap_mean_interval(sample, repetitions=2000, seed=20)
print(f"95% percentile bootstrap interval: [{bootstrap_interval[0]:.2f}, {bootstrap_interval[1]:.2f}]")

95% percentile bootstrap interval: [125.44, 141.77]


In [5]:
coverage_rng = random.Random(2040)
covered = 0
display_intervals = []
for repetition in range(1000):
    current = coverage_rng.sample(population, 40)
    current_mean = statistics.mean(current)
    current_se = statistics.stdev(current) / math.sqrt(40)
    low, high = normal_interval(current_mean, current_se)
    contains = low <= true_mean <= high
    covered += contains
    if repetition < 50:
        display_intervals.append((current_mean, low, high, contains))
print(f"Intervals covering the population mean: {covered} of 1000")
print(f"Simulated coverage: {100 * covered / 1000:.1f}%")

Intervals covering the population mean: 950 of 1000
Simulated coverage: 95.0%


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for index, (_, low, high, contains) in enumerate(display_intervals):
    axes[0].plot([low, high], [index, index], color="#25705A" if contains else "#C34A36")
axes[0].axvline(true_mean, color="black", linestyle="--", label="population mean")
axes[0].set(title="Fifty repeated 95% intervals", xlabel="Weekly sales (Rp million)", ylabel="Sample")
axes[0].legend()
sizes = [20, 40, 80, 160]
margins = [1.96 * statistics.pstdev(population) / math.sqrt(n) for n in sizes]
axes[1].plot(sizes, margins, marker="o", color="#345995")
axes[1].set(title="Margin shrinks with square root of n", xlabel="Sample size", ylabel="Approximate margin")
fig.tight_layout()
plt.show()

<Figure size 1000x400 with 2 Axes>

In [7]:
for size, margin in zip(sizes, margins):
    print(f"n={size:3d}: margin={margin:.2f}")

n= 20: margin=13.88
n= 40: margin=9.82
n= 80: margin=6.94
n=160: margin=4.91


## Your turn
Choose a confidence level and construct an interval for a business measure. State the parameter, method, assumptions, and decision threshold before interpreting it.